In [1]:
import sys
sys.path.append('..')

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from src.explain import build_baseline

X_counts = np.load('../data/features/X_counts_full_bgl.npy')
y = np.load('../data/features/y_full_bgl.npy')

X_normal = X_counts[y == 0]
contamination = float(np.clip(y.mean(), 0.001, 0.5))
print("Training on:", X_normal.shape, "| contamination:", round(contamination, 4))

Training on: (42677, 1124) | contamination: 0.1012


In [2]:
pca = PCA(n_components=5, random_state=42)
pca.fit(X_normal)

reconstructed = pca.inverse_transform(pca.transform(X_normal))
errors = np.mean((X_normal - reconstructed) ** 2, axis=1)
pca_threshold = float(np.percentile(errors, 100 * (1 - contamination)))

joblib.dump(pca, '../models/bgl/pca.joblib')
print("PCA threshold:", round(pca_threshold, 6))

PCA threshold: 8.752005


In [3]:
iso = IsolationForest(contamination=contamination, random_state=42)
iso.fit(X_normal)
joblib.dump(iso, '../models/bgl/isolation_forest.joblib')

rng = np.random.default_rng(42)
idx = rng.choice(len(X_normal), size=min(50_000, len(X_normal)), replace=False)
lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination, novelty=True)
lof.fit(X_normal[idx])
joblib.dump(lof, '../models/bgl/lof.joblib')

print("Saved all three unsupervised models")

Saved all three unsupervised models


In [ ]:

labeled = pd.read_csv('../data/parsed/bgl_labeled_lines.csv')
counts = labeled['label'].value_counts()
keep_labels = counts[counts >= 5].index
dropped = counts[counts < 5]
labeled = labeled[labeled['label'].isin(keep_labels)]

print("Classes kept:", labeled['label'].nunique(), "| rows:", len(labeled))
print("Classes dropped (< 5 examples):", dropped.index.tolist())

Classes kept: 32 | rows: 348443
Classes dropped (< 5 examples): ['KERNFLOAT', 'KERNRTSA', 'MMCS', 'MONNULL', 'LINKBLL', 'KERNEXT', 'KERNBIT', 'MONILL', 'KERNTLBE']


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    labeled['content'], labeled['label'],
    test_size=0.2, stratify=labeled['label'], random_state=42)

cause_classifier = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)),
])
cause_classifier.fit(X_train, y_train)

pred = cause_classifier.predict(X_test)
print(classification_report(y_test, pred, zero_division=0))

joblib.dump(cause_classifier, '../models/bgl/cause_classifier.joblib')

              precision    recall  f1-score   support

    APPALLOC       1.00      1.00      1.00        29
     APPBUSY       1.00      1.00      1.00       102
    APPCHILD       1.00      1.00      1.00        64
      APPOUT       1.00      1.00      1.00       163
     APPREAD       1.00      1.00      1.00      1197
      APPRES       1.00      1.00      1.00       474
      APPSEV       1.00      1.00      1.00      9930
       APPTO       1.00      1.00      1.00       398
    APPTORUS       1.00      1.00      1.00         2
     APPUNAV       1.00      1.00      1.00       410
     KERNCON       1.00      1.00      1.00         3
    KERNDTLB       1.00      1.00      1.00     30547
      KERNMC       1.00      1.00      1.00        68
   KERNMICRO       1.00      1.00      1.00       301
     KERNMNT       1.00      1.00      1.00       144
    KERNMNTF       1.00      1.00      1.00      6306
   KERNNOETH       1.00      1.00      1.00         3
     KERNPAN       1.00    

['../models/bgl/cause_classifier.joblib']

In [6]:
templates = pd.read_csv('../data/parsed/templates_full_bgl.csv', keep_default_na=False)
event_names = templates['EventId'].tolist()

baseline = build_baseline(X_counts, y)
np.save('../models/bgl/baseline.npy', baseline)

metadata = {
    'log_type': 'bgl',
    'event_names': [int(e) for e in event_names],
    'templates': {int(r.EventId): r.EventTemplate for r in templates.itertuples()},
    'pca_threshold': pca_threshold,
    'anomaly_rate': float(y.mean()),
    'trained_windows': int(len(X_normal)),
    'window_size': 100,
    'cause_classes': sorted(labeled['label'].unique().tolist()),
}

with open('../models/bgl/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata with", len(event_names), "events and",
      len(metadata['cause_classes']), "cause classes")

Saved metadata with 1124 events and 32 cause classes
